# NorthStar Urban Mobility - Query Optimisation
## Part 5: Indexing Strategies, Explain Plans & Performance Benchmarking

This notebook provides a dedicated treatment of query optimisation across both SQLite (relational) and MongoDB (NoSQL), demonstrating how indexing, query restructuring, and explain plans improve performance on the NorthStar dataset.

## Setup
Run this cell first to clone the data repository and install dependencies.

In [ ]:
import os
if not os.path.exists('northstar-coursework'):
    !git clone https://github.com/Erucard/northstar-coursework.git
!pip install -q pymongo[srv] dnspython
os.chdir('/content/northstar-coursework/data/raw')
print('Ready:', sorted([f for f in os.listdir('.') if f.endswith('.csv')]))

In [ ]:
%load_ext rpy2.ipython

In [ ]:
%%R
install.packages(c('RSQLite', 'DBI', 'readr', 'microbenchmark'), repos='https://cloud.r-project.org')

## Section A: SQLite Query Optimisation

We load the NorthStar data into SQLite, then compare query execution plans and timing before and after index creation.

In [ ]:
%%R
library(RSQLite)
library(DBI)
library(readr)
library(microbenchmark)

# Zone standardisation
standardise_zone <- function(z) {
  z <- trimws(z)
  mapping <- c('north'='North','NORTH'='North','south'='South','SOUTH'='South',
               'east'='East','EAST'='East','west'='West','WEST'='West',
               'central'='Central','CENTRAL'='Central','Ctr'='Central',
               'airport'='Airport','AIRPORT'='Airport',
               'riverside'='Riverside','RiverSide'='Riverside')
  ifelse(z %in% names(mapping), mapping[z], z)
}

data_path <- '.'

# Load data
hubs <- read_csv(paste0(data_path, '/hubs.csv'), show_col_types=FALSE)
customers <- read_csv(paste0(data_path, '/customers.csv'), show_col_types=FALSE)
customers$home_zone <- standardise_zone(customers$home_zone)
drivers <- read_csv(paste0(data_path, '/drivers.csv'), show_col_types=FALSE)
drivers$base_zone <- standardise_zone(drivers$base_zone)
vehicles <- read_csv(paste0(data_path, '/vehicles.csv'), show_col_types=FALSE)
vehicles$assigned_zone <- standardise_zone(vehicles$assigned_zone)
orders <- read_csv(paste0(data_path, '/orders.csv'), show_col_types=FALSE)
orders$pickup_zone <- standardise_zone(orders$pickup_zone)
orders$dropoff_zone <- standardise_zone(orders$dropoff_zone)
deliveries <- read_csv(paste0(data_path, '/deliveries.csv'), show_col_types=FALSE)
incidents <- read_csv(paste0(data_path, '/incidents.csv'), show_col_types=FALSE)
complaints <- read_csv(paste0(data_path, '/complaints.csv'), show_col_types=FALSE)
app_events <- read_csv(paste0(data_path, '/app_events.csv'), show_col_types=FALSE)
app_events$zone_context <- standardise_zone(app_events$zone_context)

con <- dbConnect(RSQLite::SQLite(), ':memory:')
dbWriteTable(con, 'hubs', hubs, overwrite=TRUE)
dbWriteTable(con, 'customers', customers, overwrite=TRUE)
dbWriteTable(con, 'drivers', drivers, overwrite=TRUE)
dbWriteTable(con, 'vehicles', vehicles, overwrite=TRUE)
dbWriteTable(con, 'orders', orders, overwrite=TRUE)
dbWriteTable(con, 'deliveries', deliveries, overwrite=TRUE)
dbWriteTable(con, 'incidents', incidents, overwrite=TRUE)
dbWriteTable(con, 'complaints', complaints, overwrite=TRUE)
dbWriteTable(con, 'app_events', app_events, overwrite=TRUE)

cat('Database loaded. No indexes yet.\n')

In [ ]:
%%R
# ============================================================
# TEST QUERY 1: Hub Performance with Multi-Table JOIN
# This is the most complex query - joins deliveries, orders,
# hubs, and incidents.
# ============================================================

test_query_1 <- "
SELECT h.hub_name, h.zone, COUNT(*) AS total,
       SUM(CASE WHEN d.delivery_status='Failed' THEN 1 ELSE 0 END) AS failed,
       AVG(d.fuel_or_charge_cost) AS avg_cost
FROM deliveries d
JOIN orders o ON d.order_id = o.order_id
JOIN hubs h ON d.hub_id = h.hub_id
WHERE d.delivery_status IN ('Failed','Delayed')
GROUP BY h.hub_name, h.zone
ORDER BY failed DESC
"

cat('=== BEFORE INDEXING: Query Plan for Test Query 1 ===\n')
plan_before_1 <- dbGetQuery(con, paste('EXPLAIN QUERY PLAN', test_query_1))
print(plan_before_1)

# Benchmark before indexing
bench_before_1 <- microbenchmark(
  dbGetQuery(con, test_query_1),
  times = 50
)
cat('\nBenchmark BEFORE indexing (50 runs):\n')
print(summary(bench_before_1))

In [ ]:
%%R
# ============================================================
# TEST QUERY 2: Repeat Complainers with Delivery Status
# Joins complaints, orders, and deliveries.
# ============================================================

test_query_2 <- "
SELECT c.customer_id, COUNT(DISTINCT cp.complaint_id) AS complaints,
       SUM(CASE WHEN d.delivery_status='OnTime' THEN 1 ELSE 0 END) AS ontime_complaints
FROM customers c
JOIN complaints cp ON c.customer_id = cp.customer_id
LEFT JOIN deliveries d ON cp.order_id = d.order_id
GROUP BY c.customer_id
HAVING COUNT(DISTINCT cp.complaint_id) >= 2
ORDER BY complaints DESC
"

cat('=== BEFORE INDEXING: Query Plan for Test Query 2 ===\n')
plan_before_2 <- dbGetQuery(con, paste('EXPLAIN QUERY PLAN', test_query_2))
print(plan_before_2)

bench_before_2 <- microbenchmark(
  dbGetQuery(con, test_query_2),
  times = 50
)
cat('\nBenchmark BEFORE indexing (50 runs):\n')
print(summary(bench_before_2))

In [ ]:
%%R
# ============================================================
# TEST QUERY 3: Point Lookup - Single Delivery with Incidents
# Tests index efficiency on primary key-style lookups.
# ============================================================

test_query_3 <- "
SELECT d.*, i.incident_type, i.severity
FROM deliveries d
LEFT JOIN incidents i ON d.delivery_id = i.delivery_id
WHERE d.delivery_id = 'DL00100'
"

cat('=== BEFORE INDEXING: Query Plan for Test Query 3 ===\n')
plan_before_3 <- dbGetQuery(con, paste('EXPLAIN QUERY PLAN', test_query_3))
print(plan_before_3)

bench_before_3 <- microbenchmark(
  dbGetQuery(con, test_query_3),
  times = 100
)
cat('\nBenchmark BEFORE indexing (100 runs):\n')
print(summary(bench_before_3))

In [ ]:
%%R
# ============================================================
# INDEX CREATION WITH JUSTIFICATIONS
# ============================================================

cat('=== Creating Indexes ===\n\n')

# Primary key-style indexes (unique lookups)
dbExecute(con, 'CREATE UNIQUE INDEX idx_deliveries_pk ON deliveries(delivery_id)')
cat('1. idx_deliveries_pk (delivery_id UNIQUE)\n')
cat('   Purpose: O(1) lookup for individual delivery records.\n')
cat('   Used by: Point queries, incident joins on delivery_id.\n\n')

# Foreign key indexes (JOIN acceleration)
dbExecute(con, 'CREATE INDEX idx_deliveries_order ON deliveries(order_id)')
cat('2. idx_deliveries_order (order_id)\n')
cat('   Purpose: Accelerates the most common JOIN (orders <-> deliveries).\n')
cat('   Used by: Virtually every analytical query.\n\n')

dbExecute(con, 'CREATE INDEX idx_deliveries_hub ON deliveries(hub_id)')
cat('3. idx_deliveries_hub (hub_id)\n')
cat('   Purpose: Hub-level performance queries avoid full scan.\n\n')

dbExecute(con, 'CREATE INDEX idx_deliveries_driver ON deliveries(driver_id)')
cat('4. idx_deliveries_driver (driver_id)\n')
cat('   Purpose: Driver performance analysis queries.\n\n')

dbExecute(con, 'CREATE INDEX idx_deliveries_vehicle ON deliveries(vehicle_id)')
cat('5. idx_deliveries_vehicle (vehicle_id)\n')
cat('   Purpose: Vehicle fleet analysis queries.\n\n')

dbExecute(con, 'CREATE INDEX idx_complaints_customer ON complaints(customer_id)')
cat('6. idx_complaints_customer (customer_id)\n')
cat('   Purpose: Customer complaint history lookups.\n\n')

dbExecute(con, 'CREATE INDEX idx_complaints_order ON complaints(order_id)')
cat('7. idx_complaints_order (order_id)\n')
cat('   Purpose: Linking complaints to specific orders/deliveries.\n\n')

dbExecute(con, 'CREATE INDEX idx_incidents_delivery ON incidents(delivery_id)')
cat('8. idx_incidents_delivery (delivery_id)\n')
cat('   Purpose: Incident lookup per delivery - used in every incident analysis.\n\n')

dbExecute(con, 'CREATE INDEX idx_orders_customer ON orders(customer_id)')
cat('9. idx_orders_customer (customer_id)\n')
cat('   Purpose: Customer order history lookups.\n\n')

# Composite indexes (covering queries)
dbExecute(con, 'CREATE INDEX idx_del_status_hub ON deliveries(delivery_status, hub_id)')
cat('10. idx_del_status_hub COMPOSITE (delivery_status, hub_id)\n')
cat('    Purpose: Covers the WHERE + GROUP BY in hub failure queries.\n')
cat('    The composite index satisfies both the filter and the grouping\n')
cat('    without accessing the main table (covering index).\n\n')

dbExecute(con, 'CREATE INDEX idx_del_status_zone ON deliveries(delivery_status, hub_id, fuel_or_charge_cost)')
cat('11. idx_del_status_zone_cost COMPOSITE (delivery_status, hub_id, fuel_or_charge_cost)\n')
cat('    Purpose: Fully covers Test Query 1 - the query planner can answer\n')
cat('    the WHERE filter, GROUP BY, and AVG(fuel_or_charge_cost) entirely\n')
cat('    from the index without touching the base table.\n\n')

cat('Total indexes created: 11\n')

In [ ]:
%%R
# ============================================================
# AFTER INDEXING: Query Plans and Benchmarks
# ============================================================

cat('=== AFTER INDEXING: Query Plan for Test Query 1 ===\n')
plan_after_1 <- dbGetQuery(con, paste('EXPLAIN QUERY PLAN', test_query_1))
print(plan_after_1)

bench_after_1 <- microbenchmark(
  dbGetQuery(con, test_query_1),
  times = 50
)
cat('\nBenchmark AFTER indexing (50 runs):\n')
print(summary(bench_after_1))

cat('\n=== AFTER INDEXING: Query Plan for Test Query 2 ===\n')
plan_after_2 <- dbGetQuery(con, paste('EXPLAIN QUERY PLAN', test_query_2))
print(plan_after_2)

bench_after_2 <- microbenchmark(
  dbGetQuery(con, test_query_2),
  times = 50
)
cat('\nBenchmark AFTER indexing (50 runs):\n')
print(summary(bench_after_2))

cat('\n=== AFTER INDEXING: Query Plan for Test Query 3 ===\n')
plan_after_3 <- dbGetQuery(con, paste('EXPLAIN QUERY PLAN', test_query_3))
print(plan_after_3)

bench_after_3 <- microbenchmark(
  dbGetQuery(con, test_query_3),
  times = 100
)
cat('\nBenchmark AFTER indexing (100 runs):\n')
print(summary(bench_after_3))

In [ ]:
%%R
# ============================================================
# PERFORMANCE COMPARISON SUMMARY
# ============================================================

cat('=== PERFORMANCE COMPARISON: BEFORE vs AFTER INDEXING ===\n\n')

cat(sprintf('Query 1 (Hub Performance - 3-table JOIN):\n'))
cat(sprintf('  Before: median %.2f ms\n', median(bench_before_1$time) / 1e6))
cat(sprintf('  After:  median %.2f ms\n', median(bench_after_1$time) / 1e6))
cat(sprintf('  Improvement: %.1fx faster\n\n', 
            median(bench_before_1$time) / median(bench_after_1$time)))

cat(sprintf('Query 2 (Repeat Complainers - 3-table JOIN with HAVING):\n'))
cat(sprintf('  Before: median %.2f ms\n', median(bench_before_2$time) / 1e6))
cat(sprintf('  After:  median %.2f ms\n', median(bench_after_2$time) / 1e6))
cat(sprintf('  Improvement: %.1fx faster\n\n', 
            median(bench_before_2$time) / median(bench_after_2$time)))

cat(sprintf('Query 3 (Point Lookup with Incident JOIN):\n'))
cat(sprintf('  Before: median %.2f ms\n', median(bench_before_3$time) / 1e6))
cat(sprintf('  After:  median %.2f ms\n', median(bench_after_3$time) / 1e6))
cat(sprintf('  Improvement: %.1fx faster\n\n', 
            median(bench_before_3$time) / median(bench_after_3$time)))

dbDisconnect(con)

## Section B: MongoDB Query Optimisation

We analyse MongoDB explain plans and demonstrate how compound indexes improve aggregation pipeline performance.

In [ ]:
import pymongo
from pymongo import MongoClient, ASCENDING, DESCENDING
import time

# Connect to MongoDB Atlas
MONGO_URI = 'mongodb+srv://<username>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority'
client = MongoClient(MONGO_URI)
db = client['northstar_mobility']

print('Connected to MongoDB Atlas')

In [ ]:
# ============================================================
# MONGODB EXPLAIN PLAN: Before Optimal Indexing
# ============================================================

# First, drop non-essential indexes to test without them
# (keeping _id index which cannot be dropped)
for idx_name in list(db.delivery_operations.index_information().keys()):
    if idx_name != '_id_':
        try:
            db.delivery_operations.drop_index(idx_name)
        except:
            pass

print('=== EXPLAIN PLAN: Find Query WITHOUT Indexes ===')

# Test query: Find failed deliveries at Central zone hubs
explain_before = db.delivery_operations.find(
    {'delivery_status': 'Failed', 'hub.zone': 'Central'}
).explain()

exec_stats = explain_before.get('executionStats', {})
print(f"Winning plan: {explain_before.get('queryPlanner', {}).get('winningPlan', {}).get('stage', 'N/A')}")
print(f"Documents examined: {exec_stats.get('totalDocsExamined', 'N/A')}")
print(f"Documents returned: {exec_stats.get('nReturned', 'N/A')}")
print(f"Execution time: {exec_stats.get('executionTimeMillis', 'N/A')} ms")
print(f"Scan type: COLLSCAN (full collection scan - no index available)")

In [ ]:
# ============================================================
# MONGODB INDEX CREATION WITH JUSTIFICATIONS
# ============================================================

print('=== Creating MongoDB Indexes ===\n')

# Index 1: Single field - delivery_status
db.delivery_operations.create_index('delivery_status')
print('1. delivery_status (single field)')
print('   Supports: Status filtering (WHERE delivery_status = "Failed")\n')

# Index 2: Compound - delivery_status + hub.zone
db.delivery_operations.create_index([
    ('delivery_status', ASCENDING),
    ('hub.zone', ASCENDING)
])
print('2. (delivery_status, hub.zone) compound')
print('   Supports: Hub failure analysis filtered by status\n')

# Index 3: Compound - hub.hub_id + delivery_status
db.delivery_operations.create_index([
    ('hub.hub_id', ASCENDING),
    ('delivery_status', ASCENDING)
])
print('3. (hub.hub_id, delivery_status) compound')
print('   Supports: Hub performance aggregation pipeline\n')

# Index 4: delivery_id unique
db.delivery_operations.create_index('delivery_id', unique=True)
print('4. delivery_id (unique)')
print('   Supports: Point lookups by delivery ID\n')

# Customer journeys indexes
db.customer_journeys.create_index('customer_id', unique=True)
db.customer_journeys.create_index('metrics.total_complaints')
db.customer_journeys.create_index([
    ('metrics.is_repeat_complainer', ASCENDING),
    ('engagement.loyalty_score', ASCENDING)
])
print('5-7. Customer journey indexes (customer_id, complaints, risk compound)\n')

# App event indexes
db.app_event_stream.create_index('event_type')
db.app_event_stream.create_index([('event_type', ASCENDING), ('success', ASCENDING)])
print('8-9. App event indexes (event_type, compound event+success)\n')

print('Total MongoDB indexes created: 9')

In [ ]:
# ============================================================
# MONGODB EXPLAIN PLAN: After Indexing
# ============================================================

print('=== EXPLAIN PLAN: Find Query WITH Indexes ===')

explain_after = db.delivery_operations.find(
    {'delivery_status': 'Failed', 'hub.zone': 'Central'}
).explain()

exec_stats = explain_after.get('executionStats', {})
winning = explain_after.get('queryPlanner', {}).get('winningPlan', {})

print(f"Winning plan stage: {winning.get('stage', 'N/A')}")
if 'inputStage' in winning:
    print(f"Input stage: {winning['inputStage'].get('stage', 'N/A')}")
    if 'indexName' in winning['inputStage']:
        print(f"Index used: {winning['inputStage']['indexName']}")
print(f"Documents examined: {exec_stats.get('totalDocsExamined', 'N/A')}")
print(f"Keys examined: {exec_stats.get('totalKeysExamined', 'N/A')}")
print(f"Documents returned: {exec_stats.get('nReturned', 'N/A')}")
print(f"Execution time: {exec_stats.get('executionTimeMillis', 'N/A')} ms")

print('\nThe compound index (delivery_status, hub.zone) enables IXSCAN')
print('instead of COLLSCAN, examining only matching documents.')

In [ ]:
# ============================================================
# MONGODB AGGREGATION PIPELINE EXPLAIN
# ============================================================

print('=== EXPLAIN: Hub Performance Aggregation Pipeline ===')

pipeline = [
    {'$match': {'delivery_status': {'$in': ['Failed', 'Delayed']}}},
    {'$group': {
        '_id': '$hub.hub_id',
        'total': {'$sum': 1},
        'avg_cost': {'$avg': '$route.fuel_or_charge_cost'}
    }},
    {'$sort': {'total': -1}}
]

explain_agg = db.command('aggregate', 'delivery_operations', pipeline=pipeline, explain=True)

# Navigate the explain output
stages = explain_agg.get('stages', [{}])
if stages:
    first_stage = stages[0]
    cursor = first_stage.get('$cursor', {})
    query_plan = cursor.get('queryPlanner', {})
    winning = query_plan.get('winningPlan', {})
    print(f"First stage: {winning.get('stage', 'N/A')}")
    if 'inputStage' in winning:
        print(f"Uses index: {winning['inputStage'].get('indexName', 'N/A')}")
    
print('\nThe $match stage uses the delivery_status index to filter before')
print('grouping, reducing the documents entering the $group stage.')

In [ ]:
# ============================================================
# MONGODB PERFORMANCE BENCHMARKING
# ============================================================

import time

def benchmark_query(collection, query_filter, n_runs=50):
    """Benchmark a find query over n_runs and return median time in ms."""
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        list(collection.find(query_filter))  # Force evaluation
        elapsed = (time.perf_counter() - start) * 1000
        times.append(elapsed)
    times.sort()
    return {
        'median_ms': times[len(times)//2],
        'mean_ms': sum(times)/len(times),
        'min_ms': times[0],
        'max_ms': times[-1]
    }

print('=== MongoDB Query Benchmarks (with indexes) ===\n')

# Benchmark 1: Status + zone filter
b1 = benchmark_query(db.delivery_operations, 
                     {'delivery_status': 'Failed', 'hub.zone': 'Central'})
print(f'Query: Failed deliveries at Central zone')
print(f'  Median: {b1["median_ms"]:.2f} ms, Mean: {b1["mean_ms"]:.2f} ms\n')

# Benchmark 2: Point lookup
b2 = benchmark_query(db.delivery_operations,
                     {'delivery_id': 'DL00100'}, n_runs=100)
print(f'Query: Point lookup by delivery_id')
print(f'  Median: {b2["median_ms"]:.2f} ms, Mean: {b2["mean_ms"]:.2f} ms\n')

# Benchmark 3: Customer risk query
b3 = benchmark_query(db.customer_journeys,
                     {'metrics.is_repeat_complainer': True, 'engagement.loyalty_score': {'$gte': 50}})
print(f'Query: High-loyalty repeat complainers')
print(f'  Median: {b3["median_ms"]:.2f} ms, Mean: {b3["mean_ms"]:.2f} ms\n')

# Benchmark 4: App event failure query
b4 = benchmark_query(db.app_event_stream,
                     {'event_type': 'chat_escalated', 'success': False})
print(f'Query: Failed chat_escalated events')
print(f'  Median: {b4["median_ms"]:.2f} ms, Mean: {b4["mean_ms"]:.2f} ms')

In [ ]:
# ============================================================
# COVERED QUERY DEMONSTRATION
# A covered query returns results entirely from the index
# without accessing the documents themselves.
# ============================================================

print('=== Covered Query Demonstration ===')

# Create an index that covers the projection
db.delivery_operations.create_index([
    ('delivery_status', ASCENDING),
    ('hub.hub_id', ASCENDING),
    ('route.fuel_or_charge_cost', ASCENDING)
])

# Query that only requests indexed fields + projection excludes _id
explain_covered = db.delivery_operations.find(
    {'delivery_status': 'Failed'},
    {'hub.hub_id': 1, 'route.fuel_or_charge_cost': 1, '_id': 0}
).explain()

exec_stats = explain_covered.get('executionStats', {})
print(f"Total keys examined: {exec_stats.get('totalKeysExamined', 'N/A')}")
print(f"Total docs examined: {exec_stats.get('totalDocsExamined', 'N/A')}")
print(f"Documents returned: {exec_stats.get('nReturned', 'N/A')}")

if exec_stats.get('totalDocsExamined', 1) == 0:
    print('\nCOVERED QUERY: totalDocsExamined = 0 confirms the query was')
    print('answered entirely from the index without touching documents.')
else:
    print('\nNote: MongoDB may still examine documents for nested field projections.')
    print('The compound index still reduces I/O by limiting the scan to matching keys.')

## Query Optimisation: Summary

### SQLite Optimisation Results

| Query | Type | Before | After | Improvement |
|-------|------|--------|-------|-------------|
| Hub Performance (3-table JOIN) | Analytical | Full scan on all tables | Index scans on join + filter columns | Significant |
| Repeat Complainers (JOIN + HAVING) | Analytical | Sequential scan | Index-accelerated joins | Significant |
| Point Lookup (delivery + incidents) | Transactional | Full scan | Unique index lookup | Most improved |

### MongoDB Optimisation Results

| Index Type | Purpose | Query Pattern |
|-----------|---------|---------------|
| Single field (delivery_status) | Filter stage acceleration | `$match` in aggregation pipelines |
| Compound (status + zone) | Multi-condition filter | Find queries with two equality conditions |
| Compound (event_type + success) | Covered query potential | App event failure analysis |
| Unique (delivery_id) | Point lookups | Single document retrieval |

### Key Principles Demonstrated

1. **Index columns used in JOINs** — `order_id`, `delivery_id`, `customer_id` are the most impactful indexes because they accelerate the most common operations.

2. **Composite indexes follow the ESR rule** (Equality, Sort, Range) — put equality conditions first for maximum selectivity.

3. **Covered queries** eliminate document/row access entirely by returning results from the index alone.

4. **MongoDB's document model reduces the need for joins** — by embedding related data, the delivery_operations collection eliminates 4 JOINs that the equivalent SQL query requires.

5. **EXPLAIN QUERY PLAN** (SQLite) and **explain()** (MongoDB) are essential diagnostic tools for verifying that indexes are actually being used.